In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)

In [2]:
train_url = "https://www.dropbox.com/scl/fi/m2yj95tccxmzin3mnna47/atis_train_data.csv?rlkey=p61rpu2mwxjcb1ypfh89qzull&st=omw3dkpu&dl=1"
test_url = "https://www.dropbox.com/scl/fi/d1zwrv2jslo7j93p68l75/atis_test_data.csv?rlkey=0kl75tt54i2ccau9m1fxevlhn&st=w7n0orza&dl=1"

In [3]:
df_train = pd.read_csv(train_url, index_col=0)
df_test = pd.read_csv(test_url, index_col=0)

In [4]:
df_train.head()

,query,intent,slot filling
0,i want to fly from boston at 838 am and arriv...,flight,O O O O O B-fromloc.city_name O B-depart_time...
1,what flights are available from pittsburgh to...,flight,O O O O O B-fromloc.city_name O B-toloc.city_...
2,what is the arrival time in san francisco for...,flight_time,O O O B-flight_time I-flight_time O B-fromloc...
3,cheapest airfare from tacoma to orlando,airfare,B-cost_relative O O B-fromloc.city_name O B-t...
4,round trip fares from pittsburgh to philadelp...,airfare,B-round_trip I-round_trip O O B-fromloc.city_...


In [5]:
pd.set_option('display.max_colwidth', None)

df_small = pd.DataFrame(columns=['query','intent','slot filling'])
j = 0
for i in df_train.intent.unique():
    df_small.loc[j] = df_train[df_train.intent==i].iloc[0]
    j = j + 1

In [6]:
df_small

,query,intent,slot filling
0,i want to fly from boston at 838 am and arrive in denver at 1110 in the morning,flight,O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day
1,what is the arrival time in san francisco for the 755 am flight leaving washington,flight_time,O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name
2,cheapest airfare from tacoma to orlando,airfare,B-cost_relative O O B-fromloc.city_name O B-toloc.city_name
3,what kind of aircraft is used on a flight from cleveland to dallas,aircraft,O O O O O O O O O O B-fromloc.city_name O B-toloc.city_name
4,what kind of ground transportation is available in denver,ground_service,O O O O O O O O B-city_name
5,what 's the airport at orlando,airport,O O O O O B-city_name
6,which airline serves denver pittsburgh and atlanta,airline,O O O B-fromloc.city_name B-fromloc.city_name O B-fromloc.city_name
7,how far is it from orlando airport to orlando,distance,O O O O O B-fromloc.airport_name I-fromloc.airport_name O B-toloc.city_name
8,what is fare code h,abbreviation,O O O O B-fare_basis_code
9,how much does the limousine service cost within pittsburgh,ground_fare,O O O O B-transport_type O O O B-city_name


In [7]:
df_small[['query', 'slot filling']]

,query,slot filling
0,i want to fly from boston at 838 am and arrive in denver at 1110 in the morning,O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day
1,what is the arrival time in san francisco for the 755 am flight leaving washington,O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name
2,cheapest airfare from tacoma to orlando,B-cost_relative O O B-fromloc.city_name O B-toloc.city_name
3,what kind of aircraft is used on a flight from cleveland to dallas,O O O O O O O O O O B-fromloc.city_name O B-toloc.city_name
4,what kind of ground transportation is available in denver,O O O O O O O O B-city_name
5,what 's the airport at orlando,O O O O O B-city_name
6,which airline serves denver pittsburgh and atlanta,O O O B-fromloc.city_name B-fromloc.city_name O B-fromloc.city_name
7,how far is it from orlando airport to orlando,O O O O O B-fromloc.airport_name I-fromloc.airport_name O B-toloc.city_name
8,what is fare code h,O O O O B-fare_basis_code
9,how much does the limousine service cost within pittsburgh,O O O O B-transport_type O O O B-city_name


In [8]:
query_data_train = df_train['query'].values
slot_data_train = df_train['slot filling'].values

query_data_test = df_test['query'].values
slot_data_test = df_test['slot filling'].values

In [9]:
max_query_length = 30

In [10]:
# Textvec of query

# define text_vectorization layer
text_vectorization_query = keras.layers.TextVectorization(output_sequence_length=max_query_length)

# run training corpus through layer to create vocab
text_vectorization_query.adapt(query_data_train)
query_vocab_size = text_vectorization_query.vocabulary_size()

In [11]:
text_vectorization_query.vocabulary_size()

888

In [12]:
text_vectorization_query.get_vocabulary()[:20]

['',
 '[UNK]',
 np.str_('to'),
 np.str_('from'),
 np.str_('flights'),
 np.str_('the'),
 np.str_('on'),
 np.str_('what'),
 np.str_('me'),
 np.str_('flight'),
 np.str_('boston'),
 np.str_('show'),
 np.str_('san'),
 np.str_('i'),
 np.str_('denver'),
 np.str_('a'),
 np.str_('francisco'),
 np.str_('in'),
 np.str_('and'),
 np.str_('atlanta')]

In [13]:
# vectorize train and test queries
source_train = text_vectorization_query(query_data_train)
source_test = text_vectorization_query(query_data_test)

In [14]:
df_small['slot filling'].head()

0     O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day 
1          O O O B-flight_time I-flight_time O B-fromloc.city_name I-fromloc.city_name O O B-depart_time.time I-depart_time.time O O B-fromloc.city_name 
2                                                                                            B-cost_relative O O B-fromloc.city_name O B-toloc.city_name 
3                                                                                            O O O O O O O O O O B-fromloc.city_name O B-toloc.city_name 
4                                                                                                                            O O O O O O O O B-city_name 
Name: slot filling, dtype: str

In [15]:
# Textvec of slots
text_vectorization_slots = keras.layers.TextVectorization(output_sequence_length=max_query_length, standardize=None)
text_vectorization_slots.adapt(slot_data_train)
slot_vocab_size = text_vectorization_slots.vocabulary_size()

target_train = text_vectorization_slots(slot_data_train)
target_test = text_vectorization_slots(slot_data_test)

In [16]:
text_vectorization_slots.get_vocabulary()

['',
 '[UNK]',
 np.str_('O'),
 np.str_('B-toloc.city_name'),
 np.str_('B-fromloc.city_name'),
 np.str_('I-toloc.city_name'),
 np.str_('B-depart_date.day_name'),
 np.str_('B-airline_name'),
 np.str_('I-fromloc.city_name'),
 np.str_('B-depart_time.period_of_day'),
 np.str_('I-airline_name'),
 np.str_('B-depart_date.day_number'),
 np.str_('B-depart_date.month_name'),
 np.str_('B-depart_time.time'),
 np.str_('B-round_trip'),
 np.str_('B-cost_relative'),
 np.str_('I-round_trip'),
 np.str_('B-flight_mod'),
 np.str_('B-depart_time.time_relative'),
 np.str_('I-depart_time.time'),
 np.str_('B-stoploc.city_name'),
 np.str_('B-city_name'),
 np.str_('B-class_type'),
 np.str_('B-arrive_time.time'),
 np.str_('B-arrive_time.time_relative'),
 np.str_('I-class_type'),
 np.str_('B-flight_stop'),
 np.str_('I-arrive_time.time'),
 np.str_('B-airline_code'),
 np.str_('I-depart_date.day_number'),
 np.str_('I-fromloc.airport_name'),
 np.str_('B-fromloc.airport_name'),
 np.str_('B-arrive_date.day_name'),
 np.s

In [17]:
text_vectorization_slots.vocabulary_size()

125

In [18]:
# adapted from https://keras.io/examples/nlp/text_classification_with_transformer/

from keras import ops
from keras import layers

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads,
                                             key_dim=embed_dim,
                                             output_shape=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=maxlen, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [19]:
# Params
embed_dim = 512
dense_dim = 64
num_heads = 5

units = 128

In [20]:
embedding = TokenAndPositionEmbedding(max_query_length, query_vocab_size, embed_dim)

In [21]:
te = TransformerEncoder(embed_dim, dense_dim, num_heads)

In [22]:
inputs = keras.layers.Input(shape=(max_query_length, ))

x = embedding(inputs) #gray vectors

encoder_out = te(x) #encoder_out = BLUE!!! vectors

# Classifier at the end
x = keras.layers.Dense(128, activation='relu')(encoder_out)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(slot_vocab_size, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 30, 512)        │       470,016 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 30, 512)        │     5,319,232 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 30, 128)        │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30, 125)        │        16,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,871,037 (22.40 MB)

 Trainable params: 5,871,037 (22.40 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
(num_heads * ((embed_dim + 1) * embed_dim * 3) +   # self-attention heads
(num_heads * embed_dim + 1) * embed_dim +          # concatenate-and-project
embed_dim * 2 +                                    # first layer norm
(embed_dim + 1) * dense_dim +                      # first feed-forward
(dense_dim + 1) * (embed_dim) +                    # final feed-forward
 embed_dim * 2)                                    # final layer norm

5319232

In [24]:
embedding.weights

[<Variable path=token_and_position_embedding/embedding/embeddings, shape=(888, 512), dtype=float32, value=[[ 0.03581891  0.01626286 -0.01412892 ...  0.0452905  -0.02311034
    0.02508745]
  [ 0.02021731  0.01918722 -0.00385682 ...  0.02363205 -0.02280219
   -0.00295401]
  [-0.01289467  0.03753278 -0.00539821 ...  0.02464176 -0.01119621
    0.01759655]
  ...
  [ 0.02409312 -0.00869781 -0.01521801 ...  0.00323968  0.01492134
   -0.01377813]
  [ 0.04109884  0.0002197   0.01844469 ... -0.04935136 -0.02678368
   -0.01112056]
  [-0.00839518 -0.02063971  0.00734846 ...  0.01073778  0.00010662
    0.02275636]]>,
 <Variable path=token_and_position_embedding/embedding_1/embeddings, shape=(30, 512), dtype=float32, value=[[ 0.04791559 -0.00179821  0.02660979 ...  0.02429681 -0.02299124
   -0.01301686]
  [-0.0365486  -0.01160147 -0.04650236 ... -0.03935713 -0.04289023
    0.04146255]
  [-0.02161982  0.01293555 -0.02727276 ... -0.02790877 -0.04306101
    0.04120204]
  ...
  [-0.00329853 -0.01233319 

In [25]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["sparse_categorical_accuracy"])

In [26]:
BATCH_SIZE = 64
epochs = 10

# Fit
history = model.fit(source_train, target_train, batch_size=BATCH_SIZE, epochs=epochs)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 29s 350ms/step - loss: 0.4964 - sparse_categorical_accuracy: 0.9002
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 28s 360ms/step - loss: 0.1332 - sparse_categorical_accuracy: 0.9614
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 29s 368ms/step - loss: 0.0924 - sparse_categorical_accuracy: 0.9722
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 30s 382ms/step - loss: 0.0648 - sparse_categorical_accuracy: 0.9814
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 30s 385ms/step - loss: 0.0458 - sparse_categorical_accuracy: 0.9871
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 29s 371ms/step - loss: 0.0349 - sparse_categorical_accuracy: 0.9903
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 29s 373ms/step - loss: 0.0270 - sparse_categorical_accuracy: 0.9924
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 30s 390ms/step - loss: 0.0211 - sparse_categorical_accuracy: 0.9938
Epoch 9/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 32s 404ms/step - loss: 0.0178 - sparse_categorical_accuracy: 0.9949
Epoch 10/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 34s 442

In [27]:
model.evaluate(source_test, target_test)

28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - loss: 0.0628 - sparse_categorical_accuracy: 0.9892


[0.06278040260076523, 0.9892123341560364]

In [28]:
# Metrics
def slot_filling_accuracy(actual, predicted, only_slots=False):
    not_padding = np.not_equal(actual, 0) #+ np.not_equal(predicted, 0)

    if only_slots:
        non_slot_token = text_vectorization_slots(['0']).numpy()[0, 0]
        slots = np.not_equal(actual, non_slot_token)
        correct_predictions = np.equal(actual, predicted)[not_padding * slots]
    else:
        correct_predictions = np.equal(actual, predicted)[not_padding]

    sample_length = len(correct_predictions)
    weights = np.ones(sample_length)
    return np.dot(correct_predictions, weights) / sample_length

In [29]:
predicted = np.argmax(model.predict(source_test), axis=-1).reshape(-1)
actual = target_test.numpy().reshape(-1)

acc = slot_filling_accuracy(actual, predicted, only_slots=False)
acc_slots = slot_filling_accuracy(actual, predicted, only_slots=True)

print(f'Accuracy = {acc:.3f}')
print(f'Accuracy on slots = {acc_slots:.3f}')

28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step
Accuracy = 0.969
Accuracy on slots = 0.969


In [30]:
def predict_slots_query(query):
    sentence = text_vectorization_query([query])

    prediction = np.argmax(model.predict(sentence), axis=-1)[0]

    inverse_vocab = dict(enumerate(text_vectorization_slots.get_vocabulary()))
    decoded_prediction = " ".join(inverse_vocab[int(i)] for i in prediction)
    return decoded_prediction

In [31]:
examples = [
            'from los angeles',
            'to los angeles',
            'cheapest flight from boston to los angeles tomorrow',
            'what is the airport at orlando',
            'what are the air restrictions on flights from pittsburgh to atlanta for the airfare of 416 dollars',
            'flight from boston to santiago',
]

for e in examples:
    print(e)
    print(predict_slots_query(e))
    print()

from los angeles
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
O B-fromloc.city_name I-fromloc.city_name                           

to los angeles
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
O B-fromloc.city_name I-toloc.city_name                           

cheapest flight from boston to los angeles tomorrow
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
B-cost_relative O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name B-depart_date.today_relative                      

what is the airport at orlando
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
O O O O O B-city_name                        

what are the air restrictions on flights from pittsburgh to atlanta for the airfare of 416 dollars
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
O O O O O O O O B-fromloc.city_name O B-toloc.city_name O O O O B-fare_amount I-fare_amount             

flight from boston to santiago
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
O O B-fromloc.city_name O B-toloc.city_name                         



In [32]:
predict_slots_query("Your query here!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


'O O O                           '

In [33]:
predict_slots_query("cheapest flight to fly from MIT to Mars")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


'B-cost_relative O O O O O O B-toloc.city_name                      '